# Health AI — train the food classifier

**What this does:** downloads the **Khana** dataset (~131K images, 80 Indian-dish classes,
500×500), builds a validated/deduped manifest, makes stratified splits, and fine-tunes a
mobile student model (`mobilenetv4_conv_medium`, timm) with AMP + EMA + per-epoch checkpoints
on Colab's free GPU.

**Licence honesty (read once):** Khana is licensed for **non-commercial research/education
only** and its images are web/delivery-platform sourced — the model trained here is a
**baseline/experiment**. Before any commercial release, re-train on self-collected,
consented photos (details in `ml/DATA_LICENSES.md` in the repo).

**Setup:** Runtime ▸ Change runtime type ▸ **T4 GPU**. Checkpoints go to your Google Drive
every epoch, so a disconnect costs at most one epoch. Run the cells **top to bottom, in
order** — later cells depend on variables from earlier ones.

In [1]:
# 1 — GPU sanity check
import shutil, subprocess
if shutil.which('nvidia-smi'):
    print(subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
        capture_output=True, text=True,
    ).stdout.strip())
else:
    print('NO GPU — set Runtime > Change runtime type > T4 GPU, then re-run')

Tesla T4, 15360 MiB


In [2]:
# 2 — get the repo (fresh or up-to-date) and install light dependencies
import os, sys
from pathlib import Path
import subprocess

REPO = Path('/content/Health-AI')
if not REPO.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/anshuldhiman-ai/Health-AI.git', str(REPO)],
        check=True,
    )
# Always align the clone with GitHub (handles our history rewrites too):
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)
subprocess.run(
    ['git', '-C', str(REPO), 'reset', '--hard', 'origin/main'], check=True
)

ML_SRC = str(REPO / 'ml' / 'src')
sys.path.insert(0, ML_SRC)   # notebook kernel: in-notebook imports (cell 7)
os.environ['PYTHONPATH'] = ML_SRC   # subprocesses (cells 4 & 6)

# torch + torchvision ship with Colab; we add the model zoo + dataset/eval deps
subprocess.run(['pip', 'install', '-q', 'timm', 'gdown', 'torchmetrics', 'tqdm'], check=True)

import timm, torch
print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0))
print('repo head:', subprocess.run(
    ['git', '-C', str(REPO), 'rev-parse', '--short', 'HEAD'],
    capture_output=True, text=True).stdout.strip())

torch 2.11.0+cu130 | GPU: Tesla T4
repo head: f5c48f4


## 3 — dataset
Khana ships as a Google Drive bundle of class folders (checked 2026-10-19:
[github.com/prabhuomkar/Khana](https://github.com/prabhuomkar/Khana), arXiv 2509.06006).
`gdown` pulls the whole folder (≈14 GB); extraction takes a few minutes. The files inside
have **no extensions** — that is expected, not corruption.

In [6]:
pip install kagglehub

In [7]:
import kagglehub

kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [9]:
import os

# Paste your Kaggle username and token here (directly replace the content inside the quotes)
os.environ['KAGGLE_USERNAME'] = "nsul07"
os.environ['KAGGLE_KEY'] = "KGAT_0eea7020429bb3389d11fd515f8489a9"

import kagglehub

path = kagglehub.dataset_download("kashyap077/indian-food-images-for-model-fine-tuning-2026")
print("Dataset path:", path)

100%|██████████| 11.9G/11.9G [02:20<00:00, 91.0MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/kashyap077/indian-food-images-for-model-fine-tuning-2026/versions/1


In [14]:
from torch.utils.data import Subset
from collections import Counter

class_counts = Counter(full_ds.targets)
keep_classes = {c for c, n in class_counts.items() if n >= 100}
indices = [i for i, t in enumerate(full_ds.targets) if t in keep_classes]
filtered_ds = Subset(full_ds, indices)
print("Categories after filtering:", len(keep_classes), "Images:", len(filtered_ds))

Categories after filtering: 167 Images: 151662


In [22]:
ls -la /content/Health-AI/ml/scripts/

total 24
drwxr-xr-x 2 root root 4096 Oct 20 13:15 ./
drwxr-xr-x 7 root root 4096 Oct 20 13:15 ../
-rw-r--r-- 1 root root 4071 Oct 20 13:15 export_tflite.py
-rw-r--r-- 1 root root 1713 Oct 20 13:15 make_splits.py
-rw-r--r-- 1 root root 1314 Oct 20 13:15 prepare_dataset.py
-rw-r--r-- 1 root root 3365 Oct 20 13:15 train.py


In [34]:
import csv
import random
from pathlib import Path

DATA_ROOT = Path("/root/.cache/kagglehub/datasets/kashyap077/indian-food-images-for-model-fine-tuning-2026/versions/1/Images")
OUT_CSV = Path("/content/Health-AI/ml/data/manifest_splits.csv")
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)

random.seed(42)
rows = []

# Collect all images grouped by category
for class_dir in sorted(DATA_ROOT.iterdir()):
    if not class_dir.is_dir():
        continue
    imgs = [p for p in class_dir.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
    random.shuffle(imgs)
    n = len(imgs)
    n_val = max(1, int(0.2 * n))
    for i, p in enumerate(imgs):
        split = "val" if i < n_val else "train"
        rows.append({"path": str(p), "label": class_dir.name, "split": split})

with open(OUT_CSV, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["path", "label", "split"])
    writer.writeheader()
    writer.writerows(rows)

print(f"Wrote {len(rows)} rows to {OUT_CSV}")
print("Train:", sum(1 for r in rows if r["split"] == "train"))
print("Val:", sum(1 for r in rows if r["split"] == "val"))

Wrote 156428 rows to /content/Health-AI/ml/data/manifest_splits.csv
Train: 125225
Val: 31203


In [37]:
from pathlib import Path

lines = Path("/content/Health-AI/ml/scripts/train.py").read_text().splitlines()

print("=== lines mentioning 'splits' ===")
for i, l in enumerate(lines, 1):
    if "splits" in l:
        print(f"{i}: {l}")

print("\n=== lines mentioning 'csv' or 'read_csv' ===")
for i, l in enumerate(lines, 1):
    if "csv" in l.lower():
        print(f"{i}: {l}")

print("\n=== lines mentioning 'model_name' or 'model-name' ===")
for i, l in enumerate(lines, 1):
    if "model_name" in l or "model-name" in l:
        print(f"{i}: {l}")

=== lines mentioning 'splits' ===
4:   python scripts/train.py --splits data/interim/khana/manifest_splits.csv \
25: def read_classes(splits_csv: Path) -> list[str]:
26:     with open(splits_csv, newline="", encoding="utf-8") as f:
32:     ap.add_argument("--splits", type=Path, required=True)
53:     classes = read_classes(args.splits)
56:         args.splits, "train", classes, train=True, image_size=args.image_size
58:     val_ds = ManifestDataset(args.splits, "val", classes, train=False, image_size=args.image_size)

=== lines mentioning 'csv' or 'read_csv' ===
4:   python scripts/train.py --splits data/interim/khana/manifest_splits.csv \
14: import csv
25: def read_classes(splits_csv: Path) -> list[str]:
26:     with open(splits_csv, newline="", encoding="utf-8") as f:
27:         return sorted({row["label"] for row in csv.DictReader(f)})

=== lines mentioning 'model_name' or 'model-name' ===
5:       --model-name mobilenetv4_conv_medium --epochs 30 --out runs/baseline
34:     ap.add

In [48]:
import csv
import subprocess
import sys
from pathlib import Path

# ============================================================
# Config — adjust these paths if needed
# ============================================================
DATA_ROOT = Path("/root/.cache/kagglehub/datasets/kashyap077/indian-food-images-for-model-fine-tuning-2026/versions/1/Images")
PROJECT = Path("/content/Health-AI/ml")
MANIFEST_DIR = PROJECT / "data/interim/indian_food"
MANIFEST = MANIFEST_DIR / "manifest.csv"
SPLITS = MANIFEST_DIR / "manifest_splits.csv"
OUT_DIR = PROJECT / "runs/indian_food"

MODEL_NAME = "mobilenetv4_conv_medium"
EPOCHS = 20
RATIOS = ["0.8", "0.1", "0.1"]
SEED = "42"

# ============================================================
# Step 1: scan dataset -> manifest.csv (path, label)
# ============================================================
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

print("=== Step 1: scanning dataset ===")
rows = []
for class_dir in sorted(DATA_ROOT.iterdir()):
    if not class_dir.is_dir():
        continue
    for p in class_dir.rglob("*"):
        if p.suffix.lower() in {".jpg", ".jpeg", ".png"}:
            rows.append({"path": str(p), "label": class_dir.name})

with open(MANIFEST, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["path", "label"])
    w.writeheader()
    w.writerows(rows)

classes = sorted({r["label"] for r in rows})
print(f"  images: {len(rows)}  classes: {len(classes)}")
print(f"  -> {MANIFEST}")

# ============================================================
# Step 2: make_splits.py -> manifest_splits.csv
# ============================================================
print("\n=== Step 2: generating splits ===")
r = subprocess.run(
    [sys.executable, "scripts/make_splits.py",
     "--manifest", str(MANIFEST),
     "--out", str(SPLITS),
     "--ratios", *RATIOS,
     "--seed", SEED],
    cwd=str(PROJECT), capture_output=True, text=True
)
print(r.stdout)
if r.returncode != 0:
    print("STDERR:", r.stderr)
    raise SystemExit("make_splits.py failed — see error above")

# ============================================================
# Step 3: sanity check the splits CSV
# ============================================================
print("\n=== Step 3: checking splits ===")
with open(SPLITS) as f:
    reader = csv.DictReader(f)
    cols = reader.fieldnames
    print("  columns:", cols)
    counts = {}
    label_set = set()
    n = 0
    for row in reader:
        n += 1
        label_set.add(row.get("label"))
        sp = row.get("split") or row.get("is_val") or "?"
        counts[sp] = counts.get(sp, 0) + 1
print(f"  rows: {n}  labels in splits: {len(label_set)}")
print("  split counts:", counts)
if len(label_set) != len(classes):
    print(f"  WARNING: manifest has {len(classes)} classes but splits has {len(label_set)}")

# ============================================================
# Step 4: train
# ============================================================
print("\n=== Step 4: training ===")
print(f"  model={MODEL_NAME}  epochs={EPOCHS}  out={OUT_DIR}")
r = subprocess.run(
    [sys.executable, "scripts/train.py",
     "--splits", str(SPLITS),
     "--model-name", MODEL_NAME,
     "--epochs", str(EPOCHS),
     "--out", str(OUT_DIR)],
    cwd=str(PROJECT)
)
if r.returncode != 0:
    raise SystemExit("train.py failed — see error above")
print("\nDone. Checkpoints in:", OUT_DIR)

=== Step 1: scanning dataset ===
  images: 156428  classes: 239
  -> /content/Health-AI/ml/data/interim/indian_food/manifest.csv

=== Step 2: generating splits ===
{'train': 125143, 'val': 15642, 'test': 15643}


=== Step 3: checking splits ===
  columns: ['path', 'label', 'sha256', 'width', 'height', 'source', 'split']
  rows: 156428  labels in splits: 239
  split counts: {'train': 125143, 'val': 15642, 'test': 15643}

=== Step 4: training ===
  model=mobilenetv4_conv_medium  epochs=20  out=/content/Health-AI/ml/runs/indian_food


KeyboardInterrupt: 

In [52]:
%%bash
set -e

# ---------- Config ----------
REPO=/content/Health-AI
DATA=/root/.cache/kagglehub/datasets/kashyap077/indian-food-images-for-model-fine-tuning-2026/versions/1/Images
OUT_DIR=/content/drive/MyDrive/Health-AI-runs/indian_food   # change to /content/runs/indian_food if Drive not mounted

# ---------- 1. Repo + deps ----------
if [ ! -d "$REPO" ]; then
  git clone https://github.com/anshuldhiman-ai/Health-AI.git "$REPO"
fi
cd "$REPO/ml"
pip install -q uv
uv sync --quiet

# ---------- 2. Manifest from category folders ----------
python - <<'PY'
import csv, os
from pathlib import Path
DATA_ROOT = Path("/root/.cache/kagglehub/datasets/kashyap077/indian-food-images-for-model-fine-tuning-2026/versions/1/Images")
OUT = Path("/content/Health-AI/ml/data/interim/indian_food/manifest.csv")
OUT.parent.mkdir(parents=True, exist_ok=True)
rows = []
for class_dir in sorted(DATA_ROOT.iterdir()):
    if not class_dir.is_dir(): continue
    for p in class_dir.rglob("*"):
        if p.suffix.lower() in {".jpg", ".jpeg", ".png"}:
            rows.append({"path": str(p), "label": class_dir.name})
with open(OUT, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["path", "label"]); w.writeheader(); w.writerows(rows)
print(f"manifest: {len(rows)} rows, {len({r['label'] for r in rows})} classes -> {OUT}")
PY

# ---------- 3. Splits ----------
python scripts/make_splits.py \
  --manifest data/interim/indian_food/manifest.csv \
  --out data/interim/indian_food/manifest_splits.csv \
  --ratios 0.8 0.1 0.1 --seed 42

manifest: 156428 rows, 239 classes -> /content/Health-AI/ml/data/interim/indian_food/manifest.csv
{'train': 125143, 'val': 15642, 'test': 15643}


In [ ]:
!cd /content/Health-AI/ml && python scripts/train.py --splits data/interim/indian_food/manifest_splits.csv --model-name mobilenetv4_conv_medium --epochs 20 --out /content/runs/indian_food

239 classes: ['achar', 'adhirasam', 'aloo gobi', 'aloo matar', 'aloo methi']...
  epoch 1/20:   0% 1/1303 [00:09<3:32:03,  9.77s/it, gpu_mem=2.29G, loss=5.815]/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
  epoch 1/20:   1% 14/1303 [00:19<21:32,  1.00s/it, gpu_mem=2.29G, loss=5.815]/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
  epoch 1/20:  63% 819/1303 [12:16<05:54,  1.37it/s, gpu_mem=2.36G, loss=1.786]/usr/local/lib/python3.13/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 2. 
  warnings.warn(str(msg))
  epoch 1/20: 100% 1303/1303 [19:27<00:00,  1.12it/s, gpu_mem=2.36G, loss=1.356]
val:  13% 22/163 [00:17<02:12,  1.06it/s, top1=0.604]/usr/local/lib/python3.13/di